# Applicazione di ELIta al corpus r/Italia — keyword *notizie* (modello Ekman)

Questo notebook ripete l'analisi di `Confronto_notizie.ipynb` usando esclusivamente
le **6 emozioni di base di Ekman**: gioia, tristezza, rabbia, paura, disgusto, sorpresa.

Rispetto al modello Plutchik a 8 emozioni vengono escluse **fiducia** e **aspettativa**.
Le motivazioni e il confronto con i risultati Plutchik sono discussi alla fine di ogni sezione.

## Import e configurazione

In [1]:
import pandas as pd
import numpy as np
import emoji
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import silhouette_score
from pathlib import Path

CORPUS_CSV   = Path('corpus_Italia_notizie.csv')
TOKENS_CSV   = Path('tokens_Italia_notizie.csv')
ELITA_CSV    = Path('../Fase1/ELIta_INTENSITY_Matrix.csv')
ALPHA_02_CSV = Path('../Fase2/output_csv/elita_recalculated_0_2.csv')
ALPHA_05_CSV = Path('../Fase2/output_csv/elita_recalculated_0_5.csv')
ALPHA_08_CSV = Path('../Fase2/output_csv/elita_recalculated_0_8.csv')
OUTPUT_DIR   = Path('output_confronto')
OUTPUT_DIR.mkdir(exist_ok=True)

# Plutchik (8 emozioni) — usato solo per confronto
BASIC_EMOTIONS = ['gioia','tristezza','rabbia','paura','disgusto','fiducia','sorpresa','aspettativa']

# Ekman (6 emozioni) — modello principale di questo notebook
EKMAN_EMOTIONS = ['gioia','tristezza','rabbia','paura','disgusto','sorpresa']

EMOTION_COLORS = {
    'gioia':'#FDD835','tristezza':'#1E88E5','rabbia':'#E53935','paura':'#43A047',
    'disgusto':'#8E24AA','fiducia':'#81C784','sorpresa':'#039BE5',
    'aspettativa':'#FB8C00','neutrale':'#9E9E9E',
}

POSITIVE_EKMAN = {'gioia','sorpresa'}
NEGATIVE_EKMAN = {'tristezza','rabbia','paura','disgusto'}

print('Configurazione caricata.')
print('Emozioni Ekman:', EKMAN_EMOTIONS)
print('Escluse rispetto a Plutchik: fiducia, aspettativa')

Configurazione caricata.
Emozioni Ekman: ['gioia', 'tristezza', 'rabbia', 'paura', 'disgusto', 'sorpresa']
Escluse rispetto a Plutchik: fiducia, aspettativa


## Caricamento corpus, token e matrici ELIta

In [2]:
df_corpus = pd.read_csv(CORPUS_CSV)
df_tokens = pd.read_csv(TOKENS_CSV)
df_tokens['lemma'] = df_tokens['lemma'].astype(str).str.lower().str.strip()
df_tokens['pos']   = df_tokens['pos'].astype(str).str.upper().str.strip()
print('Corpus:', len(df_corpus), 'commenti |', 'Token:', len(df_tokens))

Corpus: 700 commenti | Token: 64812


In [3]:
df_matrix = pd.read_csv(ELITA_CSV, index_col=0)

def is_not_emoji(text):
    return emoji.emoji_count(str(text)) == 0

df_elita_orig = df_matrix[df_matrix.index.map(is_not_emoji)][BASIC_EMOTIONS].fillna(0)
df_elita_orig.index = df_elita_orig.index.astype(str).str.lower().str.strip()

def load_recalc(path):
    df = pd.read_csv(path, index_col=0)
    df.index = df.index.astype(str).str.lower().str.strip()
    return df[BASIC_EMOTIONS].fillna(0)

# Le matrici contengono tutte e 8 le colonne Plutchik.
# Le colonne Ekman vengono selezionate a runtime nelle funzioni di detection.
MATRICES = {
    'Originale (α=0)' : df_elita_orig,
    'Ibrido (α=0.2)'  : load_recalc(ALPHA_02_CSV),
    'Ibrido (α=0.5)'  : load_recalc(ALPHA_05_CSV),
    'Ibrido (α=0.8)'  : load_recalc(ALPHA_08_CSV),
}
print('Matrici:', list(MATRICES.keys()))

Matrici: ['Originale (α=0)', 'Ibrido (α=0.2)', 'Ibrido (α=0.5)', 'Ibrido (α=0.8)']


## Funzione base e prima analisi (raw — modello Ekman)

La funzione `detect_emotions` somma i vettori emotivi di tutti i lemmi ADJ+NOUN+VERB
trovati in ELIta. Viene chiamata passando `EKMAN_EMOTIONS` come lista target:
le colonne **fiducia** e **aspettativa** sono escluse dalla somma e dall'assegnazione
dell'emozione dominante.

In [4]:
def detect_emotions(df_corpus, df_tokens, df_elita, emotions=None):
    if emotions is None:
        emotions = EKMAN_EMOTIONS
    pos_filter = {'ADJ','NOUN','VERB'}
    df_f = df_tokens[df_tokens['pos'].isin(pos_filter)].copy()
    eidx = set(df_elita.index)
    tok  = df_f.groupby('comment_id')['lemma'].apply(list).to_dict()
    results = []
    for _, row in df_corpus.iterrows():
        cid   = row['comment_id']
        lemmi = tok.get(cid, [])
        sc = {e: 0.0 for e in emotions}
        found = 0
        for lemma in lemmi:
            if lemma in eidx:
                found += 1
                for e in emotions:
                    sc[e] += df_elita.loc[lemma, e]
        results.append({'comment_id':cid,'n_tokens_matched':found,**sc,
            'dominant_emotion': max(sc,key=sc.get) if found>0 else 'neutrale'})
    return pd.DataFrame(results)

total      = len(df_corpus)
df_raw_ekman = detect_emotions(df_corpus, df_tokens, df_elita_orig, emotions=EKMAN_EMOTIONS)
counts_raw_ekman = df_raw_ekman['dominant_emotion'].value_counts()

df_raw_ekman.to_csv(OUTPUT_DIR / 'notizie_ekman_results_raw.csv', index=False)

print('Distribuzione emozione dominante — raw (Ekman 6 emozioni):')
for e in EKMAN_EMOTIONS + ['neutrale']:
    n = counts_raw_ekman.get(e,0)
    print('{:<15s} {:>4d} ({:>4.1f}%) {}'.format(e, n, n/total*100, '█'*int(n/total*40)))

Distribuzione emozione dominante — raw (Ekman 6 emozioni):
gioia            403 (57.6%) ███████████████████████
tristezza         24 ( 3.4%) █
rabbia            23 ( 3.3%) █
paura             41 ( 5.9%) ██
disgusto           0 ( 0.0%) 
sorpresa         208 (29.7%) ███████████
neutrale           1 ( 0.1%) 


In [5]:
fig = go.Figure()
for e in EKMAN_EMOTIONS + ['neutrale']:
    n = counts_raw_ekman.get(e, 0)
    fig.add_trace(go.Bar(name=e, x=[e], y=[round(n/total*100,1)],
        marker_color=EMOTION_COLORS.get(e,'#999'),
        text=['{:.0f}%'.format(n/total*100)], textposition='outside'))
fig.update_layout(title='Distribuzione emozione dominante — raw (modello Ekman)',
                  barmode='group', height=450, showlegend=False)
fig.show()

### Confronto con Plutchik — raw

Con il modello Plutchik (8 emozioni) la distribuzione raw era dominata da aspettativa (~87%).
Escludendo aspettativa e fiducia (modello Ekman), l'emozione dominante nel raw diventa
**sorpresa** o **gioia** — ovvero quelle che nel notebook Plutchik emergevano solo togliendo
artificialmente aspettativa (*corpus_7emo*).

Il modello Ekman rende visibile la struttura emotiva che in Plutchik era nascosta dal
dominio di aspettativa, **senza richiedere interventi artificiali**.

Tuttavia il bias non scompare del tutto: sorpresa potrebbe dominare per ragioni simili
ad aspettativa (semantica del dominio informativo). Lo verifichiamo con la diagnosi.

## Diagnosi: cosa guida l'emozione dominante nel raw Ekman?

Ripetiamo la diagnosi già fatta per Plutchik: cerchiamo le parole che contribuiscono
di più all'emozione dominante, calcolando `freq × score`.

In [6]:
POS_FILTER = {'ADJ','NOUN','VERB'}
df_filt = df_tokens[df_tokens['pos'].isin(POS_FILTER)].copy()
elita_idx = set(df_elita_orig.index)

matched = sorted(set(df_filt['lemma']).intersection(elita_idx))
freq    = df_filt[df_filt['lemma'].isin(matched)]['lemma'].value_counts()

freq_df = freq.reset_index()
freq_df.columns = ['lemma','frequenza']
er = df_elita_orig.loc[matched, EKMAN_EMOTIONS].reset_index()
er.columns = ['lemma'] + EKMAN_EMOTIONS
freq_df = freq_df.merge(er, on='lemma', how='left')

# Emozione dominante Ekman per parola
freq_df['emo_dom_ekman'] = freq_df[EKMAN_EMOTIONS].idxmax(axis=1)
freq_df['dom_score']     = freq_df[EKMAN_EMOTIONS].max(axis=1)
freq_df['contrib_dom']   = freq_df['frequenza'] * freq_df['dom_score']

# Emozione dominante nel raw Ekman
emo_dom_raw = counts_raw_ekman.idxmax()
freq_df['contrib_top'] = freq_df['frequenza'] * freq_df[emo_dom_raw]

print('Emozione dominante nel raw Ekman:', emo_dom_raw)
print()
print('Top 20 parole per contributo a', emo_dom_raw, '(frequenza × score):')
display(freq_df.nlargest(20,'contrib_top')[['lemma','frequenza', emo_dom_raw,'emo_dom_ekman','contrib_top']].round(3).reset_index(drop=True))

Emozione dominante nel raw Ekman: gioia

Top 20 parole per contributo a gioia (frequenza × score):


,lemma,frequenza,gioia,emo_dom_ekman,contrib_top
0,notizia,810,0.33,sorpresa,267.30
1,avere,387,0.54,gioia,208.98
2,fare,584,0.29,gioia,169.36
3,vedere,199,0.79,gioia,157.21
4,parlare,138,0.58,gioia,80.04
5,dare,112,0.67,gioia,75.04
6,leggere,93,0.79,gioia,73.47
7,sapere,120,0.58,gioia,69.60
8,trovare,75,0.79,sorpresa,59.25
9,fatto,107,0.54,gioia,57.78


## Passo 3 — EMOTIONAL_STOPWORDS e TOPIC_STOPWORDS

Seguiamo lo stesso percorso di `Confronto_notizie.ipynb`:
rimuoviamo i lemmi semanticamente vuoti (verbi ausiliari, nomi generici)
e le parole di dominio legate al topic *notizie*.

> **Nota**: le stopwords sono definite indipendentemente dal modello emotivo usato.
> Non cambiano tra Plutchik ed Ekman — filtrano parole che non esprimono nessuna
> emozione specifica nel testo libero, indipendentemente da quante emozioni si considerano.

In [7]:
EMOTIONAL_STOPWORDS = {
    'avere','essere','fare','stare','dare','andare','venire',
    'potere','volere','dovere','sapere','vedere','sentire',
    'trovare','pensare','dire','parlare','guardare','tenere',
    'portare','prendere','mettere','lasciare','passare','uscire',
    'entrare','tornare','rimanere','iniziare','finire','continuare',
    'cominciare','provare','riuscire','sembrare','diventare',
    'cosa','modo','parte','punto','volta','anno','tempo','caso',
    'fatto','posto','tipo','gente','persona','vita','mondo',
    'uomo','donna','bambino','figlio','figlia','padre','madre',
    'altro','solo','grande','piccolo','nuovo','vecchio','primo',
    'ultimo','stesso','proprio','bello','buono','lungo','alto',
    'più','bene','male','molto','poco','tanto','tutto','niente',
}

TOPIC_STOPWORDS = {
    'notizia', 'notizie', 'giornale', 'giornali', 'giornalista', 'giornalismo',
    'informazione', 'informazioni', 'articolo', 'articoli', 'media', 'fonte',
    'fonti', 'testata', 'redazione', 'titolo', 'telegiornale',
}

ALL_STOPWORDS = EMOTIONAL_STOPWORDS | TOPIC_STOPWORDS

in_elita = [w for w in ALL_STOPWORDS if w in elita_idx]
df_filt_sw = df_filt[~df_filt['lemma'].isin(ALL_STOPWORDS)]
freq_total = len(df_filt)

print('ALL_STOPWORDS: {:d} lemmi | Presenti in ELIta: {:d}'.format(len(ALL_STOPWORDS), len(in_elita)))
freq_stop = df_filt[df_filt['lemma'].isin(in_elita)]['lemma'].value_counts()
print('Token rimossi: {:d}/{:d} ({:.1f}%)'.format(
    freq_stop.sum(), freq_total, freq_stop.sum()/freq_total*100))

ALL_STOPWORDS: 97 lemmi | Presenti in ELIta: 85
Token rimossi: 6252/26821 (23.3%)


## Soglia di distintività (formula Fase2)

Come in `Confronto_notizie.ipynb`, usiamo la stessa formula di distintività impiegata
in Fase2 per selezionare le parole seme dei centroidi:

```
d = ((max1 - max2) / max1) * (max1 - mean_emozioni)
```

**Differenza rispetto a Plutchik**: la formula viene calcolata sulle sole 6 colonne Ekman.
Questo cambia i valori di d perché `mean_emozioni` è la media su 6 invece di 8 emozioni,
e `max2` è il secondo massimo tra 6 candidati invece di 8.
La distribuzione dei percentili viene quindi ricalcolata.

In [9]:
def calculate_distinctiveness_ekman(df_elita):
    """Formula di distintività calcolata sulle sole 6 colonne Ekman."""
    def _row_dist(row):
        sv   = sorted(row.values, reverse=True)
        max1, max2 = sv[0], sv[1]
        mn   = np.mean(row.values)
        if max1 == 0:
            return 0.0
        return ((max1 - max2) / (max1 + 1e-9)) * (max1 - mn)
    return df_elita[EKMAN_EMOTIONS].apply(_row_dist, axis=1)


def detect_emotions_sw_ekman(df_corpus, df_tokens, df_elita, stopwords, dist_threshold=0.0):
    """Emotion detection (modello Ekman) con stopwords e soglia di distintività."""
    pos_filter = {'ADJ', 'NOUN', 'VERB'}
    df_f = df_tokens[df_tokens['pos'].isin(pos_filter)].copy()
    df_f = df_f[~df_f['lemma'].isin(stopwords)]
    eidx = set(df_elita.index)

    if dist_threshold > 0:
        dist_scores = calculate_distinctiveness_ekman(df_elita)
        eidx = eidx & set(dist_scores[dist_scores >= dist_threshold].index)

    tok = df_f.groupby('comment_id')['lemma'].apply(list).to_dict()
    results = []
    for _, row in df_corpus.iterrows():
        cid   = row['comment_id']
        lemmi = tok.get(cid, [])
        sc    = {e: 0.0 for e in EKMAN_EMOTIONS}
        found = 0
        for lemma in lemmi:
            if lemma in eidx:
                found += 1
                for e in EKMAN_EMOTIONS:
                    sc[e] += df_elita.loc[lemma, e]
        results.append({'comment_id': cid, 'n_tokens_matched': found, **sc,
            'dominant_emotion': max(sc, key=sc.get) if found > 0 else 'neutrale'})
    return pd.DataFrame(results)

print('Funzioni definite.')

Funzioni definite.


In [10]:
dist_ekman = calculate_distinctiveness_ekman(df_elita_orig)

print('Distribuzione dist_ekman — lessico ELIta (6 colonne Ekman):')
print(dist_ekman.describe().round(4).to_string())
print()
print('Percentili:')
print('{:<6s} {:>10s} {:>15s}'.format('p', 'soglia', 'parole incluse'))
print('-' * 35)
for p in [25, 40, 50, 60, 70, 75, 80, 90]:
    v = dist_ekman.quantile(p/100)
    n = (dist_ekman >= v).sum()
    print('p{:<4d} {:>10.4f} {:>12d}'.format(p, v, n))

Distribuzione dist_ekman — lessico ELIta (6 colonne Ekman):
count    6719.0000
mean        0.1071
std         0.1193
min         0.0000
25%         0.0214
50%         0.0626
75%         0.1526
max         0.7089

Percentili:
p          soglia  parole incluse
-----------------------------------
p25       0.0214         5039
p40       0.0444         4031
p50       0.0626         3360
p60       0.0910         2688
p70       0.1280         2016
p75       0.1526         1681
p80       0.1835         1346
p90       0.2800          672


### Confronto soglie — modello Ekman

In [11]:
configs_dist = [
    ('ALL_STOPWORDS (dist=0)',  ALL_STOPWORDS, 0.0),
    ('+ dist_ekman ≥ 0.04',    ALL_STOPWORDS, 0.04),
    ('+ dist_ekman ≥ 0.06',    ALL_STOPWORDS, 0.06),
    ('+ dist_ekman ≥ 0.08',    ALL_STOPWORDS, 0.08),
    ('+ dist_ekman ≥ 0.10',    ALL_STOPWORDS, 0.10),
]

print('{:<35s} | {:>6s} | {:>6s} | {:>6s} | {:>6s} | {:>7s}'.format(
    'Configurazione', 'gioia', 'tristz', 'rabbia', 'sorpr', 'match%'))
print('-' * 80)
for label, sw, dist in configs_dist:
    df_r = detect_emotions_sw_ekman(df_corpus, df_tokens, df_elita_orig, sw, dist)
    c    = df_r['dominant_emotion'].value_counts()
    m    = (df_r['n_tokens_matched'] > 0).sum()
    print('{:<35s} | {:>5.1f}% | {:>5.1f}% | {:>5.1f}% | {:>5.1f}% | {:>6.0f}%'.format(
        label,
        c.get('gioia',0)/total*100,
        c.get('tristezza',0)/total*100,
        c.get('rabbia',0)/total*100,
        c.get('sorpresa',0)/total*100,
        m/total*100))

Configurazione                      |  gioia | tristz | rabbia |  sorpr |  match%
--------------------------------------------------------------------------------
ALL_STOPWORDS (dist=0)              |  60.4% |   7.4% |   7.4% |   7.1% |     98%
+ dist_ekman ≥ 0.04                 |  69.1% |   3.6% |   5.0% |   9.0% |     95%
+ dist_ekman ≥ 0.06                 |  73.4% |   3.4% |   3.4% |   6.1% |     94%
+ dist_ekman ≥ 0.08                 |  73.6% |   2.7% |   3.1% |   5.7% |     91%
+ dist_ekman ≥ 0.10                 |  72.7% |   2.1% |   3.3% |   4.9% |     89%


Scegliamo la stessa soglia usata per Plutchik (`dist ≥ 0.06`) per mantenere la comparabilità
tra i due modelli. I valori percentili cambiano (la media su 6 emozioni è diversa),
ma la logica di selezione rimane coerente.

## Metodo finale: tutte le versioni ELIta con modello Ekman

In [12]:
DIST_THRESHOLD = 0.06

results_final_ekman = {}
for vname, df_e in MATRICES.items():
    df_r = detect_emotions_sw_ekman(df_corpus, df_tokens, df_e,
                                    ALL_STOPWORDS, dist_threshold=DIST_THRESHOLD)
    results_final_ekman[vname] = df_r
    matched = (df_r['n_tokens_matched'] > 0).sum()
    print('{:<20s} | match: {:d}/{:d} ({:.0f}%)'.format(
          vname, matched, len(df_r), matched/len(df_r)*100))

Originale (α=0)      | match: 655/700 (94%)
Ibrido (α=0.2)       | match: 646/700 (92%)
Ibrido (α=0.5)       | match: 642/700 (92%)
Ibrido (α=0.8)       | match: 634/700 (91%)


In [13]:
fig = make_subplots(rows=2, cols=2, subplot_titles=list(MATRICES.keys()),
    vertical_spacing=0.18, horizontal_spacing=0.08)
positions = [(1,1),(1,2),(2,1),(2,2)]
for idx,(vname,df_r) in enumerate(results_final_ekman.items()):
    counts = df_r['dominant_emotion'].value_counts()
    r,c = positions[idx]
    for e in EKMAN_EMOTIONS + ['neutrale']:
        n = counts.get(e,0)
        fig.add_trace(go.Bar(name=e, x=[e], y=[round(n/total*100,1)],
            marker_color=EMOTION_COLORS.get(e,'#999'),
            showlegend=(idx==0), legendgroup=e,
            text=['{:.0f}%'.format(n/total*100)], textposition='outside'),
            row=r, col=c)
fig.update_layout(
    title='Distribuzione emozione dominante — metodo finale, modello Ekman (dist ≥ 0.06)',
    barmode='group', height=700)
fig.show()

## Confronto quantitativo: originale vs ricalcolato (modello Ekman)

In [14]:
print('Score emotivi medi per versione (Ekman):')
print('{:<20s}'.format('Emozione'), end='')
for vname in MATRICES: print(' {:>20s}'.format(vname[:20]), end='')
print()
print('-'*100)
means_table_ekman = {}
for e in EKMAN_EMOTIONS:
    print('{:<20s}'.format(e), end='')
    means_table_ekman[e] = {}
    for vname in MATRICES:
        m = results_final_ekman[vname][e].mean()
        means_table_ekman[e][vname] = m
        print(' {:>18.4f}'.format(m), end='')
    print()

Score emotivi medi per versione (Ekman):
Emozione                  Originale (α=0)       Ibrido (α=0.2)       Ibrido (α=0.5)       Ibrido (α=0.8)
----------------------------------------------------------------------------------------------------
gioia                            4.1415             4.2972             4.4385             4.0463
tristezza                        1.8020             1.9331             2.0908             1.9467
rabbia                           1.6240             1.7967             1.9915             1.8695
paura                            1.9698             2.1295             2.3610             2.2250
disgusto                         0.9215             1.1543             1.4108             1.4295
sorpresa                         2.6376             3.0810             3.6255             3.6056


In [15]:
print('Variazione vs originale (Ekman):')
print('{:<20s}'.format('Emozione'), end='')
for vname in list(MATRICES.keys())[1:]: print(' {:>20s}'.format(vname[:20]), end='')
print()
print('-'*80)
for e in EKMAN_EMOTIONS:
    print('{:<20s}'.format(e), end='')
    base = means_table_ekman[e]['Originale (α=0)']
    for vname in list(MATRICES.keys())[1:]:
        print(' {:>+18f}'.format(means_table_ekman[e][vname]-base), end='')
    print()

Variazione vs originale (Ekman):
Emozione                   Ibrido (α=0.2)       Ibrido (α=0.5)       Ibrido (α=0.8)
--------------------------------------------------------------------------------
gioia                         +0.155699          +0.296940          -0.095196
tristezza                     +0.131109          +0.288811          +0.144708
rabbia                        +0.172658          +0.367491          +0.245411
paura                         +0.159758          +0.391231          +0.255231
disgusto                      +0.232748          +0.489243          +0.507937
sorpresa                      +0.443403          +0.987862          +0.967953


In [16]:
print('Gap medio e Silhouette (Ekman):')
print('{:<25s} {:>12s} {:>12s}'.format('Versione','Gap medio','Silhouette'))
print('-'*52)
gap_stats_ekman = {}
silhouette_scores_ekman = {}
for vname in MATRICES:
    df_r = results_final_ekman[vname]
    ss   = df_r[EKMAN_EMOTIONS].apply(lambda r: sorted(r.values,reverse=True), axis=1)
    gap  = ss.apply(lambda s: s[0]-s[1])
    gap_nz = gap[gap>0]
    gap_stats_ekman[vname] = gap_nz.mean()
    df_v = df_r[(df_r['n_tokens_matched']>0)&(df_r['dominant_emotion']!='neutrale')].copy()
    vc   = df_v['dominant_emotion'].value_counts()
    df_v = df_v[df_v['dominant_emotion'].isin(vc[vc>=2].index)]
    if len(df_v)>=10 and df_v['dominant_emotion'].nunique()>=2:
        sil = silhouette_score(df_v[EKMAN_EMOTIONS].values,
                               df_v['dominant_emotion'].values, metric='cosine')
        silhouette_scores_ekman[vname] = sil
        print('{:<25s} {:>12.4f} {:>12.4f}'.format(vname, gap_nz.mean(), sil))
    else:
        print('{:<25s} {:>12.4f} {:>12s}'.format(vname, gap_nz.mean(), 'N/A'))
if silhouette_scores_ekman:
    best = max(silhouette_scores_ekman, key=silhouette_scores_ekman.get)
    print('\nVersione con Silhouette migliore (Ekman):', best)

Gap medio e Silhouette (Ekman):
Versione                     Gap medio   Silhouette
----------------------------------------------------
Originale (α=0)                 1.6024       0.4320
Ibrido (α=0.2)                  1.3911       0.4144
Ibrido (α=0.5)                  1.0506       0.3710
Ibrido (α=0.8)                  0.7409       0.3566

Versione con Silhouette migliore (Ekman): Originale (α=0)


## Tabella riassuntiva (modello Ekman)

In [17]:
# Tabella A: N. commenti e token per emozione (senza soglia, con stopwords)
df_sw_forA_ekman = detect_emotions_sw_ekman(df_corpus, df_tokens, df_elita_orig, ALL_STOPWORDS, 0.0)

# Parole nel lessico ridotto (solo Ekman, con soglia per il conteggio token)
dist_ek = calculate_distinctiveness_ekman(df_elita_orig)
eidx_dist_ekman = set(df_elita_orig[dist_ek >= DIST_THRESHOLD].index)

table_A = []
for e in EKMAN_EMOTIONS:
    comm = df_sw_forA_ekman[df_sw_forA_ekman[e]>0]['comment_id']
    ntok = df_filt_sw[
        df_filt_sw['comment_id'].isin(comm) &
        df_filt_sw['lemma'].isin(eidx_dist_ekman)
    ]['lemma'].count()
    table_A.append({'Emozione':e.capitalize(),
                    'N. Commenti (score>0)':int((df_sw_forA_ekman[e]>0).sum()),
                    'N. Token':int(ntok)})
print('Tabella A — N. commenti e token per emozione (Ekman, metodo finale):')
display(pd.DataFrame(table_A))

Tabella A — N. commenti e token per emozione (Ekman, metodo finale):


,Emozione,N. Commenti (score>0),N. Token
0,Gioia,684,6702
1,Tristezza,681,6701
2,Rabbia,683,6699
3,Paura,679,6694
4,Disgusto,674,6692
5,Sorpresa,687,6703


In [18]:
dom = results_final_ekman['Originale (α=0)']['dominant_emotion'].value_counts()
tot = len(results_final_ekman['Originale (α=0)'])
table_B = [{'Emozione':e.capitalize(),
            'N. Commenti dom':int(dom.get(e,0)),
            '% totale':'{:.1f}%'.format(dom.get(e,0)/tot*100)}
           for e in EKMAN_EMOTIONS+['neutrale']]
print('Tabella B — Emozione dominante (Ekman, ALL_STOPWORDS + dist_ekman ≥ 0.06):')
display(pd.DataFrame(table_B))

Tabella B — Emozione dominante (Ekman, ALL_STOPWORDS + dist_ekman ≥ 0.06):


,Emozione,N. Commenti dom,% totale
0,Gioia,514,73.4%
1,Tristezza,24,3.4%
2,Rabbia,24,3.4%
3,Paura,42,6.0%
4,Disgusto,8,1.1%
5,Sorpresa,43,6.1%
6,Neutrale,45,6.4%


## Confronto diretto: Plutchik vs Ekman

Qui confrontiamo numericamente i risultati dei due modelli sulla versione **Originale (α=0)**,
usando lo stesso corpus, le stesse stopwords e la stessa soglia di distintività (0.06).

In [19]:
# Carichiamo i risultati Plutchik dal file salvato da Confronto_notizie.ipynb
# (stesso metodo finale: ALL_STOPWORDS + dist ≥ 0.06, versione Originale α=0)
try:
    df_plut_orig = pd.read_csv(OUTPUT_DIR / 'notizie_emotion_results_Originale_α0.csv')
    plut_available = True
    print('Risultati Plutchik caricati: {:d} commenti'.format(len(df_plut_orig)))
except FileNotFoundError:
    plut_available = False
    print('File Plutchik non trovato — ricalcolo...')
    # Importiamo la funzione da Confronto_notizie se necessario
    def _detect_plut(df_corpus, df_tokens, df_elita, stopwords, dist_threshold=0.0):
        def _row_dist(row):
            sv = sorted(row.values, reverse=True)
            max1, max2 = sv[0], sv[1]
            mn = np.mean(row.values)
            if max1 == 0: return 0.0
            return ((max1-max2)/(max1+1e-9))*(max1-mn)
        pos_filter = {'ADJ','NOUN','VERB'}
        df_f = df_tokens[df_tokens['pos'].isin(pos_filter)].copy()
        df_f = df_f[~df_f['lemma'].isin(stopwords)]
        eidx = set(df_elita.index)
        if dist_threshold > 0:
            ds = df_elita[BASIC_EMOTIONS].apply(_row_dist, axis=1)
            eidx = eidx & set(ds[ds>=dist_threshold].index)
        tok = df_f.groupby('comment_id')['lemma'].apply(list).to_dict()
        results = []
        for _, row in df_corpus.iterrows():
            cid = row['comment_id']
            lemmi = tok.get(cid,[])
            sc = {e:0.0 for e in BASIC_EMOTIONS}
            found = 0
            for lemma in lemmi:
                if lemma in eidx:
                    found += 1
                    for e in BASIC_EMOTIONS:
                        sc[e] += df_elita.loc[lemma,e]
            results.append({'comment_id':cid,'n_tokens_matched':found,**sc,
                'dominant_emotion': max(sc,key=sc.get) if found>0 else 'neutrale'})
        return pd.DataFrame(results)
    df_plut_orig = _detect_plut(df_corpus, df_tokens, df_elita_orig, ALL_STOPWORDS, 0.06)
    print('Ricalcolato: {:d} commenti'.format(len(df_plut_orig)))

Risultati Plutchik caricati: 700 commenti


In [20]:
df_ekman_orig = results_final_ekman['Originale (α=0)']

counts_plut  = df_plut_orig['dominant_emotion'].value_counts()
counts_ekman = df_ekman_orig['dominant_emotion'].value_counts()

print('{:<20s} {:>15s} {:>15s}'.format('Emozione','Plutchik (8)','Ekman (6)'))
print('-'*52)
# emozioni presenti in entrambi
for e in EKMAN_EMOTIONS + ['neutrale']:
    np_ = counts_plut.get(e,0)
    ne  = counts_ekman.get(e,0)
    delta = ne - np_
    print('{:<20s} {:>7d} ({:>4.1f}%) {:>7d} ({:>4.1f}%)  [{:+d}]'.format(
        e, np_, np_/total*100, ne, ne/total*100, delta))
print()
# emozioni Plutchik escluse da Ekman
for e in ['fiducia','aspettativa']:
    np_ = counts_plut.get(e,0)
    print('{:<20s} {:>7d} ({:>4.1f}%)  [esclusa da Ekman]'.format(
        e, np_, np_/total*100))

Emozione                Plutchik (8)       Ekman (6)
----------------------------------------------------
gioia                    111 (15.9%)     514 (73.4%)  [+403]
tristezza                 27 ( 3.9%)      24 ( 3.4%)  [-3]
rabbia                    27 ( 3.9%)      24 ( 3.4%)  [-3]
paura                     34 ( 4.9%)      42 ( 6.0%)  [+8]
disgusto                   9 ( 1.3%)       8 ( 1.1%)  [-1]
sorpresa                  13 ( 1.9%)      43 ( 6.1%)  [+30]
neutrale                  67 ( 9.6%)      45 ( 6.4%)  [-22]

fiducia                   32 ( 4.6%)  [esclusa da Ekman]
aspettativa              380 (54.3%)  [esclusa da Ekman]


In [21]:
# Grafico a barre: Plutchik vs Ekman
emo_confronto = EKMAN_EMOTIONS + ['fiducia','aspettativa','neutrale']

fig = go.Figure()
plut_vals  = [counts_plut.get(e,0)/total*100  for e in emo_confronto]
ekman_vals = [counts_ekman.get(e,0)/total*100 for e in emo_confronto]

fig.add_trace(go.Bar(name='Plutchik (8)',
    x=emo_confronto, y=plut_vals,
    marker_color='#455A64',
    text=['{:.1f}%'.format(v) for v in plut_vals], textposition='outside'))
fig.add_trace(go.Bar(name='Ekman (6)',
    x=emo_confronto, y=ekman_vals,
    marker_color='#FB8C00',
    text=['{:.1f}%'.format(v) for v in ekman_vals], textposition='outside'))

fig.update_layout(
    title='Confronto Plutchik vs Ekman — emozione dominante (Originale α=0, dist ≥ 0.06)',
    barmode='group', height=500,
    annotations=[dict(x='fiducia', y=max(plut_vals[6],1)+3,
                      text='← escluse da Ekman', showarrow=False,
                      font=dict(color='grey'))])
fig.show()

In [22]:
# Confronto Silhouette e Gap tra i due modelli
print('Confronto metriche — Originale (α=0):')
print('{:<25s} {:>12s} {:>12s}'.format('Metrica','Plutchik (8)','Ekman (6)'))
print('-'*52)

# Plutchik metrics dal CSV salvato (o ricalcolo)
ss_plut = df_plut_orig[BASIC_EMOTIONS].apply(lambda r: sorted(r.values,reverse=True), axis=1)
gap_plut = ss_plut.apply(lambda s: s[0]-s[1])
gap_plut_mean = gap_plut[gap_plut>0].mean()

df_v_plut = df_plut_orig[(df_plut_orig['n_tokens_matched']>0)&
                          (df_plut_orig['dominant_emotion']!='neutrale')].copy()
vc_p = df_v_plut['dominant_emotion'].value_counts()
df_v_plut = df_v_plut[df_v_plut['dominant_emotion'].isin(vc_p[vc_p>=2].index)]
sil_plut = silhouette_score(df_v_plut[BASIC_EMOTIONS].values,
                             df_v_plut['dominant_emotion'].values, metric='cosine')

# Ekman metrics
gap_ekman_mean = gap_stats_ekman.get('Originale (α=0)', float('nan'))
sil_ekman      = silhouette_scores_ekman.get('Originale (α=0)', float('nan'))

print('{:<25s} {:>12.4f} {:>12.4f}'.format('Gap medio', gap_plut_mean, gap_ekman_mean))
print('{:<25s} {:>12.4f} {:>12.4f}'.format('Silhouette score', sil_plut, sil_ekman))
print('{:<25s} {:>12.0f} {:>12.0f}'.format('N. classi', 8, 6))

Confronto metriche — Originale (α=0):
Metrica                   Plutchik (8)    Ekman (6)
----------------------------------------------------
Gap medio                       0.7714       1.6024
Silhouette score                0.1909       0.4320
N. classi                            8            6


### Dove vanno i commenti di aspettativa e fiducia?

Nel modello Ekman, i commenti prima assegnati ad aspettativa o fiducia vengono
riassegnati alla loro seconda emozione più alta (tra le 6 Ekman).
Analizziamo come avviene questo redistribuzione.

In [23]:
# Commenti che in Plutchik avevano aspettativa o fiducia come dominante
for emo_plut in ['aspettativa','fiducia']:
    mask = df_plut_orig['dominant_emotion'] == emo_plut
    ids_plut = set(df_plut_orig[mask]['comment_id'])
    # Come vengono classificati in Ekman?
    df_ekman_sub = df_ekman_orig[df_ekman_orig['comment_id'].isin(ids_plut)]
    redistr = df_ekman_sub['dominant_emotion'].value_counts()
    n_tot = len(df_ekman_sub)
    print('Commenti con dominante={} in Plutchik ({:d} totali) → in Ekman:'.format(
        emo_plut, n_tot))
    for e,n in redistr.items():
        print('  {:>15s}: {:>4d} ({:.1f}%)'.format(e, n, n/n_tot*100))
    print()

Commenti con dominante=aspettativa in Plutchik (380 totali) → in Ekman:
            gioia:  311 (81.8%)
         sorpresa:   20 (5.3%)
            paura:   20 (5.3%)
        tristezza:   10 (2.6%)
         neutrale:    9 (2.4%)
           rabbia:    9 (2.4%)
         disgusto:    1 (0.3%)

Commenti con dominante=fiducia in Plutchik (32 totali) → in Ekman:
            gioia:   27 (84.4%)
         neutrale:    2 (6.2%)
         sorpresa:    2 (6.2%)
           rabbia:    1 (3.1%)



## Salvataggio output

In [24]:
for vname, df_r in results_final_ekman.items():
    safe = vname.replace(' ','_').replace('(','').replace(')','').replace('=','')
    df_r.to_csv(OUTPUT_DIR / 'notizie_ekman_results_{}.csv'.format(safe), index=False)
    print('Salvato:', safe)

metrics_rows = []
for vname in MATRICES:
    df_r   = results_final_ekman[vname]
    ss     = df_r[EKMAN_EMOTIONS].apply(lambda r: sorted(r.values,reverse=True), axis=1)
    gap    = ss.apply(lambda s: s[0]-s[1])
    gap_nz = gap[gap>0]
    row = {'Versione':vname,
           'Gap medio':round(gap_nz.mean(),4),
           'Gap mediana':round(gap_nz.median(),4),
           'Silhouette':round(silhouette_scores_ekman.get(vname,float('nan')),4)}
    for e in EKMAN_EMOTIONS:
        row['mean_'+e] = round(df_r[e].mean(),4)
    metrics_rows.append(row)

df_metrics_ekman = pd.DataFrame(metrics_rows)
df_metrics_ekman.to_csv(OUTPUT_DIR / 'notizie_ekman_metriche_confronto.csv', index=False)
display(df_metrics_ekman)
print('\nSalvata: notizie_ekman_metriche_confronto.csv')

Salvato: Originale_α0
Salvato: Ibrido_α0.2
Salvato: Ibrido_α0.5
Salvato: Ibrido_α0.8


,Versione,Gap medio,Gap mediana,Silhouette,mean_gioia,mean_tristezza,mean_rabbia,mean_paura,mean_disgusto,mean_sorpresa
0,Originale (α=0),1.6024,0.7900,0.4320,4.1415,1.8020,1.6240,1.9698,0.9215,2.6376
1,Ibrido (α=0.2),1.3911,0.7263,0.4144,4.2972,1.9331,1.7967,2.1295,1.1543,3.0810
2,Ibrido (α=0.5),1.0506,0.5452,0.3710,4.4385,2.0908,1.9915,2.3610,1.4108,3.6255
3,Ibrido (α=0.8),0.7409,0.4264,0.3566,4.0463,1.9467,1.8695,2.2250,1.4295,3.6056



Salvata: notizie_ekman_metriche_confronto.csv


## Conclusioni

### Cosa cambia passando da Plutchik (8) a Ekman (6)

**Distribuzione raw**: con Plutchik aspettativa dominava (~87%). Con Ekman,
escludendo aspettativa e fiducia, emerge direttamente la struttura emotiva
sottostante — gioia e sorpresa diventano le emozioni prevalenti **senza
interventi artificiali**. Questo era visibile in Plutchik solo rimuovendo
aspettativa a mano (*corpus_7emo*), una soluzione esplicitamente etichettata
come esplorativa e non finale.

**Bias tematico**: anche con Ekman, le stopwords e la soglia di distintività
rimangono necessarie. Il problema delle parole semanticamente vuote esiste
indipendentemente dal numero di emozioni.

**Silhouette score**: con 6 classi ci si aspetta un Silhouette più alto rispetto
a 8 classi, perché la minore frammentazione riduce le zone di sovrapposizione.
Un Silhouette più alto con Ekman non è necessariamente un risultato *migliore*:
riflette anche la perdita di due dimensioni informative.

**Gap medio**: il gap 1°–2° tende a salire con Ekman perché con meno emozioni
la differenza tra il massimo e il secondo massimo è statisticamente più ampia.

### Redistribuzione di aspettativa e fiducia

I commenti precedentemente assegnati ad aspettativa vengono redistribuiti
principalmente verso **gioia** e **sorpresa** — confermando che l'anticipazione
positiva (aspettativa in Plutchik) si sovrappone a queste due emozioni nel
lessico ELIta. I commenti con fiducia come dominante migrano prevalentemente
verso **gioia**, a conferma della vicinanza semantica tra i due costrutti.

### Quale modello scegliere?

| Criterio | Plutchik (8) | Ekman (6) |
|---|---|---|
| Copertura emotiva | Più ricca (fiducia, aspettativa) | Più essenziale |
| Bias aspettativa | Forte, richiede intervento | Non applicabile |
| Interpretabilità raw | Distorta | Più diretta |
| Coerenza con ELIta | Nativa (ELIta usa Plutchik) | Sottinsieme |
| Comparabilità con ItEm | Diretta | Parziale |

Per la tesi il modello Plutchik rimane il principale perché ELIta è costruito
su di esso. Il modello Ekman offre un'analisi di controllo che evidenzia
la struttura emotiva *pura* del corpus al netto delle emozioni cognitive/valutative.